In [27]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

#import pyreadstat

#df, meta = pyreadstat.read_xport('../LLCP2025.XPT')
RAW = pd.read_sas("../../data/LLCP2025.XPT ", format="xport", encoding="latin-1")
RAW = RAW.replace(5.397605346934028e-79, np.nan)
print(RAW.head(5))

RAW = RAW.rename(columns=lambda x: x.strip())

cols = RAW.columns


#print(meta.column_names)

   _STATE  FMONTH     IDATE IMONTH IDAY IYEAR  DISPCODE       SEQNO  \
0     1.0     1.0  04072025     04   07  2025    1100.0  2025000001   
1     1.0     1.0  04072025     04   07  2025    1100.0  2025000002   
2     1.0     1.0  04042025     04   04  2025    1100.0  2025000003   
3     1.0     1.0  04082025     04   08  2025    1100.0  2025000004   
4     1.0     1.0  04082025     04   08  2025    1100.0  2025000005   

           _PSU  CTELENM1  ...  _VEGRES1  _FRUTSU1  _VEGESU1  _FRTLT1A  \
0  2.025000e+09       1.0  ...       1.0     100.0     130.0       1.0   
1  2.025000e+09       1.0  ...       1.0     107.0     145.0       1.0   
2  2.025000e+09       1.0  ...       1.0      72.0     144.0       2.0   
3  2.025000e+09       1.0  ...       1.0     300.0      63.0       1.0   
4  2.025000e+09       1.0  ...       1.0     100.0     700.0       1.0   

   _VEGLT1A  _FRT16A  _VEG23A  _FRUITE1  _VEGETE1  _AIDTST4  
0       1.0      1.0      1.0       NaN       NaN       1.0  
1   

In [33]:
sub = RAW.loc[RAW["FRUITJU2"] == 555, "FTJUDA2_"]
print(sub.value_counts(dropna=False).head())

NaN    141785
Name: FTJUDA2_, dtype: int64


In [35]:
RAW["_FRUITE1"].value_counts(dropna=False)

NaN    308605
1.0     46441
2.0      1112
Name: _FRUITE1, dtype: int64

In [36]:
(RAW.select_dtypes("number") == 5.397605346934028e-79).sum().sum()

0

In [28]:
nan_counts = RAW.isna().sum() 
all_nan_cols = nan_counts[nan_counts == len(RAW)].index.tolist()
print("NaN counts per column:\n", nan_counts)
print("Columns with all NaN values:", all_nan_cols)

NaN counts per column:
 _STATE           0
FMONTH           0
IDATE            0
IMONTH           0
IDAY             0
             ...  
_FRT16A       1112
_VEG23A       1735
_FRUITE1    308605
_VEGETE1    300896
_AIDTST4     35190
Length: 283, dtype: int64
Columns with all NaN values: ['ICFQSTVR']


In [29]:
profile = pd.DataFrame({
    "dtype": RAW.dtypes,
    "n_missing": RAW.isna().sum(),
    "pct_missing": (RAW.isna().mean() * 100).round(1),
    "n_unique": RAW.nunique(),
})
profile.sort_values("pct_missing", ascending=False).to_csv("profile.csv")

In [26]:
#fruits = ['FTJUDA2_','VEGEDA2_','GRENDA1_','POTADA1_','FRNCHDA_']
#for fruit in fruits:
print(RAW["DIABETE4"].describe(),RAW["DIABETE4"].value_counts().sort_index().tail(20),'\n')
    

count    356152.000000
mean          2.739822
std           0.772203
min           1.000000
25%           3.000000
50%           3.000000
75%           3.000000
max           9.000000
Name: DIABETE4, dtype: float64 1.0     51827
2.0      2671
3.0    290712
4.0     10166
7.0       580
9.0       196
Name: DIABETE4, dtype: int64 



In [25]:
#RAW.groupby("_AGEG5YR")["_AGE80"].agg(["count", "min", "max", "mean"])
#RAW.loc[RAW["_AGEG5YR"] == 14, "_AGE80"].describe()
#RAW.assign(imputed=RAW["SEXVAR"].eq(14)).groupby("imputed")["PREGNANT"].describe()
RAW.assign(imputed=RAW["DIABETE4"].eq(14)).groupby("imputed")["PREGNANT"].describe()



,count,mean,std,min,25%,50%,75%,max
imputed,,,,,,,,
False,61396.0,2.004984,0.512002,1.0,2.0,2.0,2.0,9.0


In [31]:
#(RAW["RENTHOM1"] == 9).sum()
RAW.loc[RAW["FRUITJU2"] == 555, "FTJUDA2_"].isna().mean()

1.0

In [ ]:
keep =['_STATE','_METSTAT','_URBSTAT','SEXVAR','_AGE80',
       'MARITAL','EDUCA','RENTHOM1',
       'VETERAN3','EMPLOY1','CHILDREN','INCOME3','GENHLTH',
       'PHYSHLTH','MENTHLTH','PRIMINS2','MEDCOST1','CHECKUP1',
       'BPHIGH6','TOLDHI3','CVDINFR4','CVDCRHD4','CVDSTRK3',
       'ASTHMA3','CHCSCNC1','CHCOCNC1','CHCCOPD3','ADDEPEV3',
       'CHCKDNY2','HAVARTH4','DIABETE4','EXERANY2','PREGNANT', 
       '_BMI5','DEAF','BLIND','DECIDE','DIFFWALK','DIFFDRES',
       'DIFFALON','SLEPTIM1','_SMOKER3','USENOW3','ECIGNOW3',
       'DRNKANY6','FLUSHOT7','PNEUVAC4','TETANUS1','FRUIT2',
       'FVGREEN1','FRENCHF1','POTATOE1','VEGETAB2','HIVTST7']
df = RAW[keep].copy()

# Show first 10 rows of all columns
print(df.head(10))



In [ ]:
values_to_drop = [2,7,9]
df = df[~df["Age"].isin(values_to_drop)]

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes,
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
    "n_unique": df.nunique(),
})
profile.sort_values("pct_missing", ascending=False).to_csv("profile_post_collapse.csv")

In [16]:
pd.crosstab(RAW["FRUITJU2"], RAW["FTJUDA2_"] == 9900)

FTJUDA2_,False,True
FRUITJU2,,
101.0,28625,0
102.0,3950,0
103.0,1344,0
104.0,421,0
105.0,297,0
...,...,...
391.0,2,0
399.0,61,0
555.0,141785,0
